In [1]:
import boto3
import time
bedrock = boto3.client('bedrock', region_name='us-east-2')
response = bedrock.list_foundation_models()
print(f"Available models: {len(response['modelSummaries'])}")

Available models: 97


**Pipeline:** dinosaur knowledge base (30 species) → 150 chat examples (5 question types) → train / validation / test split → format validation (JSONL for Bedrock) → fine-tuning cost estimate → (optional) real Bedrock customization job → A/B experiment design (base vs. base + prompt vs. fine-tuned) with paired statistics.

In [2]:
import re
import json
import math
import random
import hashlib
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime
from pathlib import Path
from statistics import NormalDist

import numpy as np
import pandas as pd
from botocore.exceptions import ClientError, BotoCoreError

bedrock_runtime = boto3.client('bedrock-runtime', region_name='us-east-2')

# Models used in the A/B experiment. Prices are USD per 1M tokens (on-demand list prices, update them if they change).
BASE_MODEL_ID = 'us.anthropic.claude-haiku-4-5-20251001-v1:0'
BASE_PRICES = (1.0, 5.0)    # (input, output)
JUDGE_MODEL_ID = 'us.anthropic.claude-sonnet-4-6'
JUDGE_PRICES = (3.0, 15.0)

# Fine-tuning: Amazon Nova Micro (text-only, cheapest Nova). Customization runs in us-east-1 and the resulting
# custom model can be served on-demand (pay per token, no Provisioned Throughput). Claude 3 Haiku has no published
# fine-tuning price, so it is not used here.
FT_REGION = 'us-east-1'
FT_BASE_MODEL = 'amazon.nova-micro-v1:0:128k'
NOVA_MICRO_ID = 'amazon.nova-micro-v1:0'     # same model, on-demand, used as the un-tuned control
RUN_TRAINING = False        # True = start a real (paid, about $0.02) customization job; False = only prepare the data (job dinobot-ft-20261026-105313 already completed)
S3_BUCKET = None            # created / reused by the setup cell below (bucket in FT_REGION)
ROLE_ARN = None             # created / reused by the setup cell below (role that Bedrock assumes)
FT_MODEL_ARN = 'arn:aws:bedrock:us-east-1:964217566728:custom-model-deployment/vpn07gqve8on'   # on-demand deployment of the trained model; adds arm C to the A/B test

DATA_DIR = Path("dino_finetune")
DATA_DIR.mkdir(exist_ok=True)
TRAIN_FILE, VAL_FILE, TEST_FILE = DATA_DIR / "train.jsonl", DATA_DIR / "validation.jsonl", DATA_DIR / "test.json"
SEED = 42

**Task 1 - Dataset.** 30 species × 5 question types (period, diet, size, location, profile) = 150 examples. Facts live in one table, so every answer is consistent with it. Three phrasings per question type add variety. All answers follow the same *house style* that the fine-tuned model should learn: plain text, 1-3 short sentences, ending with a sentence that starts with `Fun fact:`. A style like this is what fine-tuning is good at; new knowledge is better served by RAG (exercise 6).

Each record uses the Bedrock Nova fine-tuning format (`bedrock-conversation-2024`): `{"schemaVersion": ..., "system": [{"text": ...}], "messages": [{"role": "user", "content": [{"text": ...}]}, {"role": "assistant", ...}]}`, one JSON object per line.

In [3]:
# --- Task 1: dataset of 100+ examples (dinosaur species) ---
# (name, period, diet, length in m, weight, region, fun fact)
SPECIES = [
    ("Tyrannosaurus rex", "Late Cretaceous (about 68-66 million years ago)", "carnivore", 12.3, "8 tonnes", "North America", "It had one of the strongest bites of any land animal that ever lived."),
    ("Triceratops", "Late Cretaceous (about 68-66 million years ago)", "herbivore", 9, "6 to 12 tonnes", "North America", "Its skull, with three horns and a bony frill, could be over 2 meters long."),
    ("Velociraptor", "Late Cretaceous (about 75-71 million years ago)", "carnivore", 2, "15 kilograms", "Mongolia", "It was about the size of a turkey and was covered in feathers."),
    ("Stegosaurus", "Late Jurassic (about 155-150 million years ago)", "herbivore", 9, "5 tonnes", "North America", "Its tail carried four spikes, nicknamed the thagomizer."),
    ("Brachiosaurus", "Late Jurassic (about 154-150 million years ago)", "herbivore", 22, "35 tonnes", "North America", "Its front legs were longer than its hind legs, giving it a giraffe-like posture."),
    ("Spinosaurus", "Cretaceous (about 99-93 million years ago)", "carnivore", 14, "7 tonnes", "North Africa", "It had a large sail on its back and hunted fish in rivers."),
    ("Ankylosaurus", "Late Cretaceous (about 68-66 million years ago)", "herbivore", 6.5, "6 tonnes", "North America", "Its tail ended in a heavy bony club used for defense."),
    ("Diplodocus", "Late Jurassic (about 154-152 million years ago)", "herbivore", 26, "15 tonnes", "North America", "Its whip-like tail may have been cracked to make loud noises."),
    ("Allosaurus", "Late Jurassic (about 155-145 million years ago)", "carnivore", 9.7, "2 tonnes", "North America", "It was the top predator of its time and is one of the best-known Jurassic theropods."),
    ("Parasaurolophus", "Late Cretaceous (about 76-73 million years ago)", "herbivore", 10, "3.5 tonnes", "North America", "Its hollow head crest probably worked like a trumpet to produce calls."),
    ("Iguanodon", "Early Cretaceous (about 126-122 million years ago)", "herbivore", 10, "3.5 tonnes", "Europe", "It was one of the first dinosaurs ever named, back in 1825."),
    ("Pachycephalosaurus", "Late Cretaceous (about 70-66 million years ago)", "herbivore", 4.5, "450 kilograms", "North America", "Its skull had a dome of solid bone up to 25 centimeters thick."),
    ("Giganotosaurus", "Late Cretaceous (about 99-97 million years ago)", "carnivore", 12.5, "7 tonnes", "Argentina", "It was among the largest meat-eating dinosaurs ever found."),
    ("Carnotaurus", "Late Cretaceous (about 72-69 million years ago)", "carnivore", 8, "1.5 tonnes", "Argentina", "It had two horns above its eyes and very short arms."),
    ("Compsognathus", "Late Jurassic (about 150 million years ago)", "carnivore", 1, "3 kilograms", "Europe", "It was about the size of a chicken, one of the smallest known dinosaurs."),
    ("Gallimimus", "Late Cretaceous (about 70 million years ago)", "omnivore", 6, "450 kilograms", "Mongolia", "Its long legs made it a fast runner, much like a modern ostrich."),
    ("Dilophosaurus", "Early Jurassic (about 193 million years ago)", "carnivore", 7, "400 kilograms", "North America", "It had two thin crests on its skull, and there is no evidence it spat venom."),
    ("Archaeopteryx", "Late Jurassic (about 150 million years ago)", "carnivore", 0.5, "1 kilogram", "Europe", "It is a famous transitional fossil between dinosaurs and birds."),
    ("Argentinosaurus", "Late Cretaceous (about 97-94 million years ago)", "herbivore", 30, "70 tonnes", "Argentina", "It is one of the heaviest land animals known to science."),
    ("Therizinosaurus", "Late Cretaceous (about 70 million years ago)", "herbivore", 10, "5 tonnes", "Mongolia", "Its hand claws could reach about 1 meter in length."),
    ("Protoceratops", "Late Cretaceous (about 75-71 million years ago)", "herbivore", 1.8, "180 kilograms", "Mongolia", "A famous fossil shows a Protoceratops locked in a fight with a Velociraptor."),
    ("Baryonyx", "Early Cretaceous (about 130-125 million years ago)", "carnivore", 10, "1.2 tonnes", "Europe", "It had a huge thumb claw and a crocodile-like snout for catching fish."),
    ("Utahraptor", "Early Cretaceous (about 126 million years ago)", "carnivore", 6, "500 kilograms", "North America", "It is the largest known member of the raptor family, with fossils found in Utah."),
    ("Coelophysis", "Late Triassic (about 215-205 million years ago)", "carnivore", 3, "20 kilograms", "North America", "Hundreds of its skeletons were discovered together at Ghost Ranch in New Mexico."),
    ("Plateosaurus", "Late Triassic (about 214-204 million years ago)", "herbivore", 8, "0.6 to 4 tonnes", "Europe", "It is one of the earliest large long-necked dinosaurs."),
    ("Mamenchisaurus", "Late Jurassic (about 160-145 million years ago)", "herbivore", 26, "20 tonnes", "Asia", "Its neck made up about half of its total body length."),
    ("Edmontosaurus", "Late Cretaceous (about 73-66 million years ago)", "herbivore", 12, "4 tonnes", "North America", "This duck-billed dinosaur had hundreds of tightly packed teeth."),
    ("Deinonychus", "Early Cretaceous (about 115-108 million years ago)", "carnivore", 3.4, "73 kilograms", "North America", "Its discovery changed the view of dinosaurs from slow reptiles to active, bird-like hunters."),
    ("Microraptor", "Early Cretaceous (about 120 million years ago)", "carnivore", 0.8, "1 kilogram", "Asia", "It had feathered wings on both its arms and its legs."),
    ("Ceratosaurus", "Late Jurassic (about 153-148 million years ago)", "carnivore", 6, "0.5 tonnes", "North America", "It had a distinctive horn on its snout and a row of small bony plates along its back."),
]

QUESTIONS = {
    "period":   ["When did {name} live?", "Which geological period was {name} from?", "What era did {name} roam the Earth in?"],
    "diet":     ["What did {name} eat?", "Was {name} a herbivore, a carnivore or an omnivore?", "What was the diet of {name}?"],
    "size":     ["How big was {name}?", "What was the size of {name}?", "How long and how heavy was {name}?"],
    "location": ["Where have {name} fossils been found?", "Where did {name} live?", "In which region was {name} discovered?"],
    "profile":  ["Tell me about {name}.", "Give me a quick profile of {name}.", "What is {name} known for?"],
}
ANSWERS = {
    "period":   "{name} lived during the {period}.",
    "diet":     "{name} was a {diet}.",
    "size":     "{name} reached about {length} m in length and weighed roughly {weight}.",
    "location": "{name} fossils have been found in {region}.",
    "profile":  "{name} lived during the {period} in {region}. It was a {diet} about {length} m long.",
}
SYSTEM_PROMPT = ("You are DinoBot, a paleontology assistant. Answer in one to three short sentences of plain text "
                 "and always finish with a sentence that starts with 'Fun fact:'.")


def build_dataset(species=SPECIES, seed=SEED):
    rng = random.Random(seed)
    records = []
    for name, period, diet, length, weight, region, fact in species:
        fields = dict(name=name, period=period, diet=diet, length=length, weight=weight, region=region)
        must_include = {"period": [period.split(" (")[0]], "diet": [diet], "size": [str(length)], "location": [region],
                        "profile": [period.split(" (")[0], diet, region]}
        for qtype, phrasings in QUESTIONS.items():
            records.append({"species": name, "type": qtype, "question": rng.choice(phrasings).format(name=name),
                            "answer": f"{ANSWERS[qtype].format(**fields)} Fun fact: {fact}",
                            "must_include": must_include[qtype]})
    return records


def to_bedrock_record(example):
    return {"schemaVersion": "bedrock-conversation-2024",
            "system": [{"text": SYSTEM_PROMPT}],
            "messages": [{"role": "user", "content": [{"text": example["question"]}]},
                         {"role": "assistant", "content": [{"text": example["answer"]}]}]}


def split_dataset(records, val_size=15, test_size=15, seed=SEED):
    shuffled = records[:]
    random.Random(seed).shuffle(shuffled)
    return shuffled[val_size + test_size:], shuffled[:val_size], shuffled[val_size:val_size + test_size]


def write_jsonl(path, records):
    path.write_text("\n".join(json.dumps(to_bedrock_record(r), ensure_ascii=False) for r in records) + "\n", encoding="utf-8")


dataset = build_dataset()
train, val, test = split_dataset(dataset)
write_jsonl(TRAIN_FILE, train)
write_jsonl(VAL_FILE, val)
TEST_FILE.write_text(json.dumps(test, ensure_ascii=False, indent=1), encoding="utf-8")   # keeps question + facts for the A/B test

assert len(dataset) >= 100
print(f"{len(dataset)} examples ({len(SPECIES)} species x {len(QUESTIONS)} types) -> train {len(train)} / validation {len(val)} / test {len(test)}")
print("Per type:", dict(Counter(r["type"] for r in dataset)))
print("\nExample record in", TRAIN_FILE.name)
print(TRAIN_FILE.read_text(encoding="utf-8").splitlines()[0])

150 examples (30 species x 5 types) -> train 120 / validation 15 / test 15
Per type: {'period': 30, 'diet': 30, 'size': 30, 'location': 30, 'profile': 30}

Example record in train.jsonl
{"schemaVersion": "bedrock-conversation-2024", "system": [{"text": "You are DinoBot, a paleontology assistant. Answer in one to three short sentences of plain text and always finish with a sentence that starts with 'Fun fact:'."}], "messages": [{"role": "user", "content": [{"text": "Which geological period was Giganotosaurus from?"}]}, {"role": "assistant", "content": [{"text": "Giganotosaurus lived during the Late Cretaceous (about 99-97 million years ago). Fun fact: It was among the largest meat-eating dinosaurs ever found."}]}]}


**Task 2 - Format validation.** `validate_file` checks a JSONL file before any money is spent and returns a report instead of raising on the first problem:
- every line is valid JSON with only the keys `schemaVersion` / `system` / `messages`, text wrapped as `[{"text": ...}]`;
- `messages` starts with `user`, alternates roles, ends with `assistant`, no empty content;
- Nova's reserved keywords (`User:`, `Bot:`, `Assistant:`, `System:`, `<image>`, `<video>`, `[EOS]`) do not appear in any prompt, otherwise the job fails;
- estimated tokens per record under the model limit (≈ 4 characters per token);
- dataset-level limits (minimum / maximum number of records) and duplicate prompts;
- project-specific *style* check (the `Fun fact:` ending) reported as a warning.

The demo at the bottom feeds it a file with deliberate mistakes to show the errors it catches.

In [4]:
# --- Task 2: data format validation ---
LIMITS = {"min_train": 8, "max_train": 20_000, "max_val": 1_000, "max_tokens_per_record": 32_000}  # Nova Micro on Bedrock
RESERVED = ["User:", "Bot:", "Assistant:", "System:", "<image>", "<video>", "[EOS]"]
CHARS_PER_TOKEN = 4


def estimate_tokens(text):
    return max(1, math.ceil(len(text) / CHARS_PER_TOKEN))


def _text_blocks(blocks):
    """'content' / 'system' must be a non-empty list of {'text': non-empty str}. Returns (texts, error)."""
    if not isinstance(blocks, list) or not blocks:
        return [], "must be a non-empty list of {'text': ...} blocks"
    if not all(isinstance(b, dict) and set(b) == {"text"} and isinstance(b["text"], str) and b["text"].strip() for b in blocks):
        return [], "every block must be exactly {'text': <non-empty string>}"
    return [b["text"] for b in blocks], None


def record_texts(rec):
    return [t for b in rec.get("system", []) for t in [b["text"]]] + [b["text"] for m in rec["messages"] for b in m["content"]]


def record_tokens(rec):
    return sum(estimate_tokens(t) for t in record_texts(rec))


def validate_record(rec):
    """Return (errors, warnings) for a single parsed record."""
    errors, warnings = [], []
    if not isinstance(rec, dict):
        return ["record is not a JSON object"], warnings
    if unknown := set(rec) - {"schemaVersion", "system", "messages"}:
        errors.append(f"unknown keys: {sorted(unknown)}")
    if not isinstance(rec.get("schemaVersion"), str):
        errors.append("'schemaVersion' (string) is required")
    if "system" in rec and (err := _text_blocks(rec["system"])[1]):
        errors.append(f"'system' {err}")
    msgs = rec.get("messages")
    if not isinstance(msgs, list) or len(msgs) < 2:
        return errors + ["'messages' must be a list with at least a user and an assistant message"], warnings
    for i, m in enumerate(msgs):
        expected = "user" if i % 2 == 0 else "assistant"
        if not isinstance(m, dict) or set(m) != {"role", "content"}:
            errors.append(f"message {i}: must have exactly 'role' and 'content'")
        elif m["role"] != expected:
            errors.append(f"message {i}: role is '{m['role']}', expected '{expected}'")
        elif err := _text_blocks(m["content"])[1]:
            errors.append(f"message {i}: content {err}")
    if isinstance(msgs[-1], dict) and msgs[-1].get("role") != "assistant":
        errors.append("last message must come from the assistant")
    if not errors:
        texts = record_texts(rec)
        if hit := [k for k in RESERVED if any(k in t for t in texts)]:
            errors.append(f"reserved keyword(s) {hit} would make the training job fail")
        if record_tokens(rec) > LIMITS["max_tokens_per_record"]:
            errors.append(f"~{record_tokens(rec)} tokens exceeds {LIMITS['max_tokens_per_record']}")
        if "Fun fact:" not in msgs[-1]["content"][0]["text"]:
            warnings.append("assistant answer lacks the 'Fun fact:' ending (house style)")
    return errors, warnings


def validate_file(path, kind="train"):
    """Validate a JSONL file. Returns {'ok', 'records', 'tokens', 'errors', 'warnings', 'duplicates'}."""
    errors, warnings, tokens, prompts = [], [], 0, Counter()
    lines = [l for l in Path(path).read_text(encoding="utf-8").splitlines() if l.strip()]
    for n, line in enumerate(lines, start=1):
        try:
            rec = json.loads(line)
        except json.JSONDecodeError as e:
            errors.append(f"line {n}: invalid JSON ({e.msg})")
            continue
        e, w = validate_record(rec)
        errors += [f"line {n}: {x}" for x in e]
        warnings += [f"line {n}: {x}" for x in w]
        if not e:
            tokens += record_tokens(rec)
            prompts[rec["messages"][0]["content"][0]["text"].strip().lower()] += 1
    duplicates = sum(c - 1 for c in prompts.values() if c > 1)
    if duplicates:
        warnings.append(f"{duplicates} duplicate prompt(s)")
    low, high = (LIMITS["min_train"], LIMITS["max_train"]) if kind == "train" else (1, LIMITS["max_val"])
    if not low <= len(lines) <= high:
        errors.append(f"{kind} file has {len(lines)} records, allowed range is {low}-{high}")
    return {"ok": not errors, "records": len(lines), "tokens": tokens, "errors": errors, "warnings": warnings, "duplicates": duplicates}


def print_report(name, report):
    print(f"{name}: {'VALID' if report['ok'] else 'INVALID'} | {report['records']} records | ~{report['tokens']:,} tokens "
          f"| {len(report['errors'])} errors | {len(report['warnings'])} warnings")
    for line in report["errors"][:8] + report["warnings"][:3]:
        print("   -", line)


reports = {"train": validate_file(TRAIN_FILE, "train"), "validation": validate_file(VAL_FILE, "validation")}
for name, rep in reports.items():
    print_report(name, rep)
assert all(r["ok"] for r in reports.values()), "fix the dataset before fine-tuning"

# The validator on a broken file: one example of each kind of mistake
bad_file = DATA_DIR / "broken_demo.jsonl"
good = to_bedrock_record(train[0])
turn = lambda role, text: {"role": role, "content": [{"text": text}]}
bad_file.write_text("\n".join(json.dumps(x) if not isinstance(x, str) else x for x in [
    good,
    "{not json",
    {**good, "extra": 1},
    {"schemaVersion": "bedrock-conversation-2024", "messages": [turn("assistant", "I start with the assistant"), turn("user", "x")]},
    {"schemaVersion": "bedrock-conversation-2024", "messages": [turn("user", "Question?"), turn("assistant", "  ")]},
    {"schemaVersion": "bedrock-conversation-2024", "messages": [{"role": "user", "content": "plain string"}, turn("assistant", "x")]},
    {"schemaVersion": "bedrock-conversation-2024", "messages": [turn("user", "Assistant: hi"), turn("assistant", "Fun fact: x")]},
    {"schemaVersion": "bedrock-conversation-2024", "messages": [turn("user", "Same question"), turn("assistant", "No style.")]},
    {"schemaVersion": "bedrock-conversation-2024", "messages": [turn("user", "Same question"), turn("assistant", "No style.")]},
]) + "\n", encoding="utf-8")
print_report("\nbroken_demo", validate_file(bad_file, "validation"))

train: VALID | 120 records | ~10,486 tokens | 0 errors | 0 warnings
validation: VALID | 15 records | ~1,305 tokens | 0 errors | 0 warnings

broken_demo: INVALID | 9 records | ~104 tokens | 8 errors | 3 warnings
   - line 2: invalid JSON (Expecting property name enclosed in double quotes)
   - line 3: unknown keys: ['extra']
   - line 4: message 0: role is 'assistant', expected 'user'
   - line 4: message 1: role is 'user', expected 'assistant'
   - line 4: last message must come from the assistant
   - line 5: message 1: content every block must be exactly {'text': <non-empty string>}
   - line 6: message 0: content must be a non-empty list of {'text': ...} blocks
   - line 7: reserved keyword(s) ['Assistant:'] would make the training job fail
   - line 8: assistant answer lacks the 'Fun fact:' ending (house style)
   - line 9: assistant answer lacks the 'Fun fact:' ending (house style)
   - 1 duplicate prompt(s)


**Task 3 - Cost estimate.** Fine-tuning has three cost components:
1. **Training** = tokens in the training file × epochs × price per 1K tokens (a one-off).
2. **Storage** of the custom model (monthly).
3. **Inference**: Nova custom models can be deployed **on-demand** (pay per token, same rate as the base model, nothing while idle). Provisioned Throughput is optional and only worth it for steady high volume.

Prices are taken from the AWS Price List API for Amazon Bedrock in us-east-1 (checked 2026-10-26): Nova Micro customization training $0.001 / 1K tokens, storage $1.95 / model / month, custom-model inference $0.000035 / 1K input and $0.00014 / 1K output tokens, Provisioned Throughput $60.50 / hour with no commitment. Token counts are estimated with ≈ 4 characters per token; the real job reports the exact number. The break-even analysis compares the fine-tuned model against the cheapest alternative that gets a similar style: Haiku 4.5 with a long few-shot prompt (arm B).

In [5]:
# --- Task 3: estimated fine-tuning cost ---
FT_PRICING = {
    "training_per_1k_tokens": 0.001,     # USD, Nova Micro customization training
    "storage_per_month": 1.95,           # USD per custom model per month
    "custom_inference_per_1m": (0.035, 0.14),   # USD per 1M (input, output) tokens, on-demand custom Nova Micro
    "provisioned_per_hour": 60.5,        # USD per model unit per hour, no commitment (optional alternative to on-demand)
}
HOURS_PER_MONTH = 730


def training_cost(tokens, epochs, price_per_1k=FT_PRICING["training_per_1k_tokens"]):
    return tokens * epochs * price_per_1k / 1000


train_tokens = reports["train"]["tokens"]
EPOCHS = 2
print(f"Training file: {train_tokens:,} estimated tokens -> {train_tokens * EPOCHS:,} tokens trained at {EPOCHS} epochs")
print(f"Training cost (one-off): ${training_cost(train_tokens, EPOCHS):.2f}")
print(f"Storage: ${FT_PRICING['storage_per_month']:.2f}/month | on-demand inference: no idle cost "
      f"(Provisioned Throughput instead would be ${FT_PRICING['provisioned_per_hour'] * HOURS_PER_MONTH:,.0f}/month 24/7)\n")

# Scenarios: how the one-off training cost grows with dataset size and epochs
avg_tokens = train_tokens / reports["train"]["records"]
scenarios = pd.DataFrame({f"{e} epoch(s)": [training_cost(avg_tokens * n, e) for n in (reports["train"]["records"], 500, 1000, 5000)]
                          for e in (1, 2, 3, 5)}, index=[f"{n:,} examples" for n in (reports["train"]["records"], 500, 1000, 5000)])
print("Training cost in USD:")
print(scenarios.round(2).to_string())


def monthly_costs(requests_per_month, prompt_tokens=30, extra_prompt_tokens=0, out_tokens=60):
    """Monthly cost of arm B (Haiku 4.5 + few-shot prompt) vs the fine-tuned Nova Micro on-demand (short prompt)."""
    in_b, out_b = BASE_PRICES
    in_c, out_c = FT_PRICING["custom_inference_per_1m"]
    base = requests_per_month * ((prompt_tokens + extra_prompt_tokens) * in_b + out_tokens * out_b) / 1_000_000
    tuned = FT_PRICING["storage_per_month"] + requests_per_month * (prompt_tokens * in_c + out_tokens * out_c) / 1_000_000
    return {"requests_per_month": requests_per_month, "base_plus_prompt": base, "finetuned_on_demand": tuned, "monthly_saving": base - tuned}


FEW_SHOT_TOKENS = sum(estimate_tokens(r["question"] + r["answer"]) for r in train[:5]) + estimate_tokens(SYSTEM_PROMPT)   # what arm B adds to every prompt
rows = [monthly_costs(n, extra_prompt_tokens=FEW_SHOT_TOKENS) for n in (10_000, 1_000_000, 50_000_000)]
print(f"\nMonthly cost, Haiku 4.5 + few-shot prompt (~{FEW_SHOT_TOKENS} extra prompt tokens/request) vs fine-tuned Nova Micro:")
print(pd.DataFrame(rows).round(2).to_string(index=False))
one_off = training_cost(train_tokens, EPOCHS)
m = monthly_costs(1_000_000, extra_prompt_tokens=FEW_SHOT_TOKENS)
saving_per_request = (m["base_plus_prompt"] - (m["finetuned_on_demand"] - FT_PRICING["storage_per_month"])) / 1_000_000
print(f"\nSaving per request: ${saving_per_request:.6f} -> the one-off training cost (${one_off:.2f}) is repaid after {math.ceil(one_off / saving_per_request):,} requests.")
print("Reading: with on-demand custom Nova the cost is tiny. The saving comes mostly from switching to a much cheaper model, so the real question is quality: does fine-tuned Nova Micro match Haiku 4.5 (arm B)? The A/B test answers that.")

Training file: 10,486 estimated tokens -> 20,972 tokens trained at 2 epochs
Training cost (one-off): $0.02
Storage: $1.95/month | on-demand inference: no idle cost (Provisioned Throughput instead would be $44,165/month 24/7)

Training cost in USD:
                1 epoch(s)  2 epoch(s)  3 epoch(s)  5 epoch(s)
120 examples          0.01        0.02        0.03        0.05
500 examples          0.04        0.09        0.13        0.22
1,000 examples        0.09        0.17        0.26        0.44
5,000 examples        0.44        0.87        1.31        2.18

Monthly cost, Haiku 4.5 + few-shot prompt (~266 extra prompt tokens/request) vs fine-tuned Nova Micro:
 requests_per_month  base_plus_prompt  finetuned_on_demand  monthly_saving
              10000              5.96                 2.04            3.92
            1000000            596.00                11.40          584.60
           50000000          29800.00               474.45        29325.55

Saving per request: $0.000587 ->

**Starting the real job.** With `RUN_TRAINING = True` this cell does everything, step by step and safe to re-run:
1. **Pre-flight**: checks that Nova Micro is fine-tunable in `FT_REGION` for your account.
2. **Resources** (`ensure_resources`): creates (or reuses) the S3 bucket `dinobot-finetune-<account>-us-east-1` with public access blocked, and the IAM role `DinoBotBedrockFineTuneRole`. The role can only be assumed by Bedrock model-customization jobs of *your* account (trust policy conditions `aws:SourceAccount` / `aws:SourceArn`) and can only read/write that one bucket.
3. **Training**: uploads the validated files and creates the customization job, then polls until it finishes (this takes a while; the notebook just waits).
4. **Deployment**: creates an on-demand custom model deployment and stores its ARN in `FT_MODEL_ARN`, so arm C joins the A/B test below.

Your AWS credentials need `iam:CreateRole`/`PutRolePolicy`/`GetRole`/`UpdateAssumeRolePolicy`/`PassRole`, S3 bucket + object permissions, and `bedrock:CreateModelCustomizationJob`/`GetModelCustomizationJob`/`CreateCustomModelDeployment`/`GetCustomModelDeployment`. Refresh them first if they expired (`aws login` or `aws sso login`).

In [6]:
# --- Training job: resources, upload, create_model_customization_job, deployment ---
def ensure_resources(region=FT_REGION, role_name="DinoBotBedrockFineTuneRole"):
    """Create (or reuse) the S3 bucket and the IAM role used by the customization job. Returns (bucket, role_arn)."""
    account = boto3.client('sts').get_caller_identity()["Account"]
    bucket = f"dinobot-finetune-{account}-{region}"
    s3 = boto3.client('s3', region_name=region)
    try:
        s3.head_bucket(Bucket=bucket)
    except ClientError as e:
        if e.response['Error']['Code'] not in ('404', 'NoSuchBucket'):
            raise
        s3.create_bucket(Bucket=bucket, **({} if region == 'us-east-1' else {"CreateBucketConfiguration": {"LocationConstraint": region}}))
        print(f"  created bucket {bucket}")
    s3.put_public_access_block(Bucket=bucket, PublicAccessBlockConfiguration={
        "BlockPublicAcls": True, "IgnorePublicAcls": True, "BlockPublicPolicy": True, "RestrictPublicBuckets": True})

    iam = boto3.client('iam')
    trust = {"Version": "2012-10-17", "Statement": [{
        "Effect": "Allow", "Principal": {"Service": "bedrock.amazonaws.com"}, "Action": "sts:AssumeRole",
        "Condition": {"StringEquals": {"aws:SourceAccount": account},
                      "ArnEquals": {"aws:SourceArn": f"arn:aws:bedrock:{region}:{account}:model-customization-job/*"}}}]}
    access = {"Version": "2012-10-17", "Statement": [
        {"Effect": "Allow", "Action": ["s3:GetObject", "s3:PutObject"], "Resource": f"arn:aws:s3:::{bucket}/*"},
        {"Effect": "Allow", "Action": ["s3:ListBucket"], "Resource": f"arn:aws:s3:::{bucket}"}]}
    try:
        role_arn = iam.get_role(RoleName=role_name)["Role"]["Arn"]
        iam.update_assume_role_policy(RoleName=role_name, PolicyDocument=json.dumps(trust))
    except iam.exceptions.NoSuchEntityException:
        role_arn = iam.create_role(RoleName=role_name, AssumeRolePolicyDocument=json.dumps(trust),
                                   Description="Lets Bedrock model customization jobs read/write the DinoBot fine-tuning bucket")["Role"]["Arn"]
        print(f"  created role {role_arn}")
        time.sleep(10)   # IAM is eventually consistent
    iam.put_role_policy(RoleName=role_name, PolicyName="DinoBotS3Access", PolicyDocument=json.dumps(access))
    return bucket, role_arn


def start_finetuning_job(bucket, role_arn, epochs=EPOCHS, learning_rate=1e-5, warmup_steps=10):
    s3 = boto3.client('s3', region_name=FT_REGION)
    stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
    for f in (TRAIN_FILE, VAL_FILE):
        s3.upload_file(str(f), bucket, f"dinobot/{f.name}")
    client = boto3.client('bedrock', region_name=FT_REGION)
    job = client.create_model_customization_job(
        jobName=f"dinobot-ft-{stamp}", customModelName=f"dinobot-{stamp}", roleArn=role_arn,
        baseModelIdentifier=FT_BASE_MODEL, customizationType="FINE_TUNING",
        trainingDataConfig={"s3Uri": f"s3://{bucket}/dinobot/{TRAIN_FILE.name}"},
        validationDataConfig={"validators": [{"s3Uri": f"s3://{bucket}/dinobot/{VAL_FILE.name}"}]},
        outputDataConfig={"s3Uri": f"s3://{bucket}/dinobot/output/"},
        hyperParameters={"epochCount": str(epochs), "learningRate": str(learning_rate), "learningRateWarmupSteps": str(warmup_steps)})
    return client, job["jobArn"]


def wait_for_job(client, job_arn, poll_seconds=60):
    while True:
        status = client.get_model_customization_job(jobIdentifier=job_arn)
        print(f"  {datetime.now():%H:%M:%S} {status['status']}")
        if status["status"] in ("Completed", "Failed", "Stopped"):
            return status
        time.sleep(poll_seconds)


def deploy_custom_model(client, model_arn, poll_seconds=30):
    """On-demand deployment of the custom model; returns the deployment ARN to use as modelId."""
    deployment = client.create_custom_model_deployment(modelDeploymentName=f"dinobot-{datetime.now():%Y%m%d-%H%M%S}", modelArn=model_arn)
    arn = deployment["customModelDeploymentArn"]
    while True:
        status = client.get_custom_model_deployment(customModelDeploymentIdentifier=arn)["status"]
        print(f"  {datetime.now():%H:%M:%S} deployment {status}")
        if status in ("Active", "Failed"):
            return arn if status == "Active" else None
        time.sleep(poll_seconds)


if RUN_TRAINING:
    client = boto3.client('bedrock', region_name=FT_REGION)
    tunable = {m["modelId"] for m in client.list_foundation_models(byCustomizationType="FINE_TUNING")["modelSummaries"]}
    assert FT_BASE_MODEL in tunable, f"{FT_BASE_MODEL} is not fine-tunable in {FT_REGION}; options: {sorted(tunable)}"
    print("1. Pre-flight OK:", FT_BASE_MODEL)
    S3_BUCKET, ROLE_ARN = ensure_resources()
    print("2. Resources:", S3_BUCKET, "|", ROLE_ARN)
    client, job_arn = start_finetuning_job(S3_BUCKET, ROLE_ARN)
    print("3. Training job started:", job_arn)
    final = wait_for_job(client, job_arn)
    print(final["status"], final.get("failureMessage", ""), "| custom model:", final.get("outputModelArn"))
    if final["status"] == "Completed":
        FT_MODEL_ARN = deploy_custom_model(client, final["outputModelArn"])
        print("4. Deployment ARN (FT_MODEL_ARN):", FT_MODEL_ARN)
else:
    print("RUN_TRAINING is False: no job started. Data is validated and ready in", DATA_DIR.resolve())

RUN_TRAINING is False: no job started. Data is validated and ready in C:\Users\MatiasSarmiento\Desktop\Onboarding\dino_finetune


**Task 4 - A/B experiment design.**

| Element | Design |
|---|---|
| **Hypothesis** | H0: the fine-tuned model is no better than the baseline. H1: it follows the DinoBot style more often and answers at least as accurately, with a shorter prompt. |
| **Arms** | **A** Haiku 4.5 zero-shot (control) · **B** Haiku 4.5 + system prompt + 5 few-shot examples (the *prompt-engineering baseline*; fine-tuning must match it at lower cost) · **N** Nova Micro with the short system prompt (the *un-tuned* model we fine-tune, so C vs N isolates the effect of fine-tuning) · **C** Nova Micro fine-tuned (added when `FT_MODEL_ARN` is set). |
| **Unit / assignment** | *Offline*: paired design, every held-out test question goes to every arm, which removes question difficulty as noise. *Online*: sticky random assignment by hashing the user id (`assign_arm`), so a user always sees the same arm. |
| **Metrics** | Primary: judge score (correctness + style, 1-5). Secondary: fact accuracy against the knowledge base, format compliance (`Fun fact:` ending, plain text, ≤ 4 sentences). Guardrails: latency, cost per request, output length. |
| **Analysis** | Paired differences vs. control: mean, 95% bootstrap CI, sign-flip permutation p-value, win/tie/loss. |
| **Sample size** | `required_sample_size` gives the number of questions needed to detect a minimum effect at 80% power. |
| **Decision rule** | Ship C only if its judge-score difference vs. N is above 0 (CI) and vs. B is not meaningfully below 0, fact accuracy is not worse, and the monthly cost from Task 3 is acceptable. Otherwise keep the prompt-engineering arm. |
| **Threats** | The judge is a model (use a different family or spot-check by hand) · test set must stay out of training · 15 questions is a pilot, not proof (see sample size). |

In [7]:
# --- Task 4: A/B harness (arms, call helper, metrics) ---
_RETRYABLE = {'ThrottlingException', 'ServiceUnavailableException', 'ModelNotReadyException'}
_runtime_clients = {}


def runtime_client(region):
    return _runtime_clients.setdefault(region, boto3.client('bedrock-runtime', region_name=region))


def call_model(model_id, prompt, system=None, max_tokens=300, max_retries=4, region='us-east-2'):
    """Single Converse call -> {'text', 'input_tokens', 'output_tokens', 'latency', 'error'}."""
    kwargs = {"modelId": model_id, "messages": [{"role": "user", "content": [{"text": prompt}]}],
              "inferenceConfig": {"maxTokens": max_tokens, "temperature": 0}}
    if system:
        kwargs["system"] = [{"text": system}]
    failure = {"text": "", "input_tokens": 0, "output_tokens": 0, "latency": None}
    for attempt in range(max_retries + 1):
        start = time.time()
        try:
            response = runtime_client(region).converse(**kwargs)
        except ClientError as e:
            code = e.response['Error']['Code']
            if code in _RETRYABLE and attempt < max_retries:
                time.sleep(2 ** attempt)
                continue
            return {**failure, "error": f"{code}: {e.response['Error']['Message']}"}
        except BotoCoreError as e:
            return {**failure, "error": f"{type(e).__name__}: {e}"}
        usage = response["usage"]
        return {"text": " ".join(b["text"] for b in response["output"]["message"]["content"] if "text" in b).strip(),
                "input_tokens": usage["inputTokens"], "output_tokens": usage["outputTokens"],
                "latency": time.time() - start, "error": None}


def cost_usd(input_tokens, output_tokens, input_price, output_price):
    return (input_tokens * input_price + output_tokens * output_price) / 1_000_000


# Few-shot examples come from the TRAIN split only (never from the test set)
FEW_SHOT_SYSTEM = SYSTEM_PROMPT + "\n\nExamples:\n" + "\n".join(f"Q: {r['question']}\nA: {r['answer']}" for r in train[:5])

NOVA_PRICES = FT_PRICING["custom_inference_per_1m"]   # custom Nova Micro is billed at the base Nova Micro rate
ARMS = {
    "A: Haiku zero-shot":   {"model_id": BASE_MODEL_ID, "system": None, "region": "us-east-2", "prices": BASE_PRICES},
    "B: Haiku + prompt":    {"model_id": BASE_MODEL_ID, "system": FEW_SHOT_SYSTEM, "region": "us-east-2", "prices": BASE_PRICES},
    "N: Nova Micro base":   {"model_id": NOVA_MICRO_ID, "system": SYSTEM_PROMPT, "region": FT_REGION, "prices": NOVA_PRICES},
}
if FT_MODEL_ARN:
    ARMS["C: Nova Micro fine-tuned"] = {"model_id": FT_MODEL_ARN, "system": SYSTEM_PROMPT, "region": FT_REGION, "prices": NOVA_PRICES}
CONTROL = "A: Haiku zero-shot"
# (treatment, baseline) pairs: everything vs. control, plus the two comparisons that matter for the fine-tuning decision
PAIRS = [(a, CONTROL) for a in ARMS if a != CONTROL] + (
    [("C: Nova Micro fine-tuned", "N: Nova Micro base"), ("C: Nova Micro fine-tuned", "B: Haiku + prompt")] if FT_MODEL_ARN else [])


def assign_arm(user_id, arms=tuple(ARMS), salt="dinobot-exp-1"):
    """Online experiments: sticky, uniform assignment by hashing the user id."""
    bucket = int(hashlib.sha256(f"{salt}:{user_id}".encode()).hexdigest(), 16) % len(arms)
    return arms[bucket]


print("Arms:", list(ARMS))
print("Assignment of 10,000 users:", dict(Counter(assign_arm(f"user-{i}") for i in range(10_000))))


# --- Metrics: fact accuracy, format compliance, LLM judge (correctness + style) ---
def split_sentences(text):
    return [s for s in re.split(r'(?<=[.!?])\s+', text.strip()) if s]


def fact_accuracy(answer, must_include):
    return sum(t.lower() in answer.lower() for t in must_include) / len(must_include)


def format_ok(answer):
    return float("Fun fact:" in answer and not re.search(r"[*#]", answer) and len(split_sentences(answer)) <= 4)


JUDGE_SYSTEM = ("You are a strict, impartial evaluator of a dinosaur Q&A assistant. "
                "Compare the candidate with the reference answer and reply with ONLY a JSON object.")


def judge(question, reference, answer):
    prompt = (f"Question: {question}\n\nReference answer: {reference}\n\nCandidate answer: {answer}\n\n"
              "Score the candidate from 1 (very poor) to 5 (excellent):\n"
              "- correctness: facts agree with the reference and are not wrong\n"
              "- style: plain text, 1-3 short sentences, ending with a sentence that starts with 'Fun fact:'\n\n"
              'Reply with JSON only: {"correctness": n, "style": n}')
    for _ in range(2):  # one retry if the reply is not valid JSON
        r = call_model(JUDGE_MODEL_ID, prompt, JUDGE_SYSTEM, max_tokens=60)
        try:
            data = json.loads(re.search(r"\{.*\}", r["text"], re.S).group(0))
            scores = {k: int(data[k]) for k in ("correctness", "style")}
            assert all(1 <= v <= 5 for v in scores.values())
            return {**scores, "judge_cost": cost_usd(r["input_tokens"], r["output_tokens"], *JUDGE_PRICES)}
        except Exception:
            continue
    return {"correctness": np.nan, "style": np.nan, "judge_cost": 0.0}


def run_arm_question(job):
    arm_name, ex = job
    arm = ARMS[arm_name]
    r = call_model(arm["model_id"], ex["question"], arm["system"], region=arm["region"])
    row = {"arm": arm_name, "question": ex["question"], "type": ex["type"], "answer": r["text"], "error": r["error"],
           "latency": r["latency"], "output_tokens": r["output_tokens"],
           "cost": cost_usd(r["input_tokens"], r["output_tokens"], *arm["prices"])}
    if r["text"]:
        row |= {"fact_accuracy": fact_accuracy(r["text"], ex["must_include"]), "format_ok": format_ok(r["text"]),
                **judge(ex["question"], ex["answer"], r["text"])}
    return row

Arms: ['A: Haiku zero-shot', 'B: Haiku + prompt', 'N: Nova Micro base', 'C: Nova Micro fine-tuned']
Assignment of 10,000 users: {'N: Nova Micro base': 2528, 'B: Haiku + prompt': 2468, 'C: Nova Micro fine-tuned': 2490, 'A: Haiku zero-shot': 2514}


**Running the experiment.** Every arm answers the 15 held-out test questions (paired design) and the judge scores each answer. `judge_score` is the mean of correctness and style.

In [8]:
# --- Run the offline paired experiment ---
test_set = json.loads(TEST_FILE.read_text(encoding="utf-8"))
jobs = [(arm, ex) for ex in test_set for arm in ARMS]
with ThreadPoolExecutor(max_workers=4) as pool:
    results = pd.DataFrame(pool.map(run_arm_question, jobs))

failed = results["error"].notna().sum()
results["judge_score"] = results[["correctness", "style"]].mean(axis=1)
print(f"{len(results)} answers from {len(ARMS)} arms" + (f" ({failed} failed)" if failed else ""))

METRICS = ["judge_score", "correctness", "style", "fact_accuracy", "format_ok", "latency", "output_tokens", "cost"]
summary = results.groupby("arm")[METRICS].mean().reindex(list(ARMS))
summary["cost_per_1k"] = summary["cost"] * 1000
print(summary.drop(columns="cost").round(3).to_string())
results[results.type == "profile"][["arm", "question", "answer"]].head(len(ARMS)).to_string(index=False)

60 answers from 4 arms
                          judge_score  correctness  style  fact_accuracy  format_ok  latency  output_tokens  cost_per_1k
arm                                                                                                                     
A: Haiku zero-shot              2.700        4.067  1.333          0.711        0.0    1.921        138.733        0.713
B: Haiku + prompt               4.533        4.400  4.667          0.578        1.0    1.424         70.600        0.703
N: Nova Micro base              4.400        3.867  4.933          0.222        1.0    0.683         40.133        0.007
C: Nova Micro fine-tuned        4.600        4.267  4.933          0.689        1.0    0.842         31.200        0.006


'                     arm                                question                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        answer\n      A: Haiku zero-sho

**Statistics.** For each metric the arms are compared with the control **question by question** (paired):
- 95% CI of the mean difference from a bootstrap over questions;
- p-value from a sign-flip permutation test (under H0 each paired difference is equally likely to be positive or negative);
- win / tie / loss counts.

`required_sample_size` uses the normal approximation `n = ((z_α/2 + z_β) · σ / δ)²`, with σ the standard deviation of the paired differences measured here and δ the smallest improvement worth detecting.

In [9]:
# --- Paired analysis and sample size ---
def paired_compare(a, b, n_boot=10_000, seed=SEED):
    """a, b: paired scores (same questions, same order). Returns the statistics of the difference b - a."""
    d = np.asarray(b, dtype=float) - np.asarray(a, dtype=float)
    d = d[~np.isnan(d)]
    rng = np.random.default_rng(seed)
    boot = rng.choice(d, size=(n_boot, len(d))).mean(axis=1)
    signs = rng.choice([-1, 1], size=(n_boot, len(d)))
    perm = np.abs((signs * d).mean(axis=1)) >= abs(d.mean()) - 1e-12
    return {"n": len(d), "mean_diff": d.mean(), "ci_low": np.percentile(boot, 2.5), "ci_high": np.percentile(boot, 97.5),
            "p_value": perm.mean(), "wins": int((d > 0).sum()), "ties": int((d == 0).sum()), "losses": int((d < 0).sum()), "sd_diff": d.std(ddof=1)}


def required_sample_size(sd_diff, min_effect, alpha=0.05, power=0.8):
    z = NormalDist().inv_cdf
    return math.ceil(((z(1 - alpha / 2) + z(power)) * sd_diff / min_effect) ** 2)


wide = {m: results.pivot(index="question", columns="arm", values=m) for m in ("judge_score", "fact_accuracy", "format_ok")}
rows = []
for arm, baseline in PAIRS:
    for metric, table in wide.items():
        rows.append({"comparison": f"{arm} vs {baseline}", "metric": metric, **paired_compare(table[baseline], table[arm])})
comparison = pd.DataFrame(rows)
print(comparison.drop(columns="sd_diff").round(3).to_string(index=False))

print("\nSample size per comparison (alpha 0.05, power 0.8) to detect a +0.5 judge-score difference:")
for _, r in comparison[comparison.metric == "judge_score"].iterrows():
    n = required_sample_size(r["sd_diff"], 0.5) if r["sd_diff"] > 0 else "n/a (no variance)"
    print(f"  {r['comparison']}: SD of differences {r['sd_diff']:.2f} -> {n} questions (this pilot used {int(r['n'])})")

                                    comparison        metric  n  mean_diff  ci_low  ci_high  p_value  wins  ties  losses
       B: Haiku + prompt vs A: Haiku zero-shot   judge_score 15      1.833   1.533    2.100    0.000    15     0       0
       B: Haiku + prompt vs A: Haiku zero-shot fact_accuracy 15     -0.133  -0.333    0.000    0.507     0    13       2
       B: Haiku + prompt vs A: Haiku zero-shot     format_ok 15      1.000   1.000    1.000    0.000    15     0       0
      N: Nova Micro base vs A: Haiku zero-shot   judge_score 15      1.700   1.367    2.067    0.000    15     0       0
      N: Nova Micro base vs A: Haiku zero-shot fact_accuracy 15     -0.489  -0.733   -0.244    0.008     0     7       8
      N: Nova Micro base vs A: Haiku zero-shot     format_ok 15      1.000   1.000    1.000    0.000    15     0       0
C: Nova Micro fine-tuned vs A: Haiku zero-shot   judge_score 15      1.900   1.467    2.333    0.000    15     0       0
C: Nova Micro fine-tuned vs A: H

**Reading the results.** Compare the table with the decision rule above. With only 15 questions the confidence intervals are wide, so treat this run as a pilot that validates the harness: generate more held-out questions (or collect real traffic with `assign_arm`) until the sample size is reached. Once training finishes and `FT_MODEL_ARN` is set, arm C is evaluated with exactly the same code, and the decision is: **does C clearly beat N (the effect of fine-tuning) and stay close to B (the better but pricier model)?**